# Entrenamiento YOLOv8 — Sistema de Monitoreo EPP (4 clases)
**Materia:** Inteligencia Artificial en Sistemas Embebidos — 2026

Dataset fusionado: construction-safety-monitor + Hard Hats — 32,134 imágenes totales

**Clases a detectar:** `helmet`, `no-helmet`, `vest`, `no-vest`

### Antes de ejecutar:
1. Ir a **Entorno de ejecución → Cambiar tipo de entorno de ejecución**
2. Seleccionar **GPU T4** y guardar
3. Subir `EPP_merged.zip` a Google Drive en la carpeta `EPP_Dataset`
4. Ejecutar las celdas en orden

## Celda 1 — Verificar que la GPU está activa

In [ ]:
!nvidia-smi

## Celda 2 — Instalar YOLOv8

In [ ]:
!pip install ultralytics -q

## Celda 3 — Conectar Google Drive y descomprimir el dataset fusionado

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile

# ============================================================
# Subir EPP_merged.zip a Google Drive en la carpeta EPP_Dataset
# ============================================================
ZIP_NAME     = 'EPP_merged.zip'
ZIP_PATH     = f'/content/drive/MyDrive/EPP_Dataset/{ZIP_NAME}'
EXTRACT_PATH = '/content/dataset'
# ============================================================

print(f'Descomprimiendo {ZIP_NAME}...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_PATH)
print('Listo. Contenido:')
os.listdir(EXTRACT_PATH)

## Celda 4 — Filtrar labels: quedarse solo con helmet, no-helmet, vest, no-vest

El dataset fusionado tiene 11 clases. Esta celda filtra los archivos de label
para conservar solo las 4 clases necesarias y remapea los IDs:

| Clase original | ID original | ID nuevo |
|---|---|---|
| helmet | 3 | 0 |
| no-helmet | 7 | 1 |
| vest | 10 | 2 |
| no-vest | 8 | 3 |

In [ ]:
import os, glob

# Mapeo: ID viejo -> ID nuevo (solo las 4 clases que nos interesan)
REMAP = {3: 0, 7: 1, 10: 2, 8: 3}

DATASET_DIR = '/content/dataset'

total_archivos = 0
total_bbox_antes = 0
total_bbox_despues = 0

for split in ['train', 'valid', 'test']:
    labels_dir = os.path.join(DATASET_DIR, split, 'labels')
    if not os.path.exists(labels_dir):
        print(f'[{split}] carpeta no encontrada, saltando')
        continue

    archivos = glob.glob(os.path.join(labels_dir, '*.txt'))
    vacios = 0

    for fpath in archivos:
        with open(fpath, 'r') as f:
            lineas = f.read().splitlines()

        total_bbox_antes += len(lineas)
        nuevas = []
        for linea in lineas:
            partes = linea.strip().split()
            if not partes:
                continue
            clase_id = int(partes[0])
            if clase_id in REMAP:
                partes[0] = str(REMAP[clase_id])
                nuevas.append(' '.join(partes))

        total_bbox_despues += len(nuevas)
        with open(fpath, 'w') as f:
            f.write('\n'.join(nuevas))
        if not nuevas:
            vacios += 1

    total_archivos += len(archivos)
    print(f'[{split}] {len(archivos)} archivos procesados — {vacios} quedaron vacíos (sin casco/chaleco)')

print(f'\nTotal bboxes antes: {total_bbox_antes}')
print(f'Total bboxes después (4 clases): {total_bbox_despues}')
print(f'Bboxes eliminados (otras clases): {total_bbox_antes - total_bbox_despues}')

## Celda 5 — Crear data.yaml con las 4 clases

In [ ]:
YAML_PATH = os.path.join(DATASET_DIR, 'data.yaml')

yaml_content = f"""train: {DATASET_DIR}/train/images
val: {DATASET_DIR}/valid/images
test: {DATASET_DIR}/test/images

nc: 4
names: ['helmet', 'no-helmet', 'vest', 'no-vest']
"""

with open(YAML_PATH, 'w') as f:
    f.write(yaml_content)

print('data.yaml creado:')
print(yaml_content)

## Celda 6 — Entrenar el modelo

Con el dataset fusionado y solo 4 clases se espera:
- **no-helmet** mejore de 28% a >65% mAP50
- **no-vest** se mantenga >66% mAP50
- Tiempo estimado en T4: ~2-3 horas (menos clases = más rápido)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=YAML_PATH,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    device=0,
    project='/content/drive/MyDrive/EPP_Dataset',
    name='modelo_epp_v3',
    exist_ok=True
)

print('\n=== ENTRENAMIENTO COMPLETADO ===')
print('Mejor modelo: /content/drive/MyDrive/EPP_Dataset/modelo_epp_v3/weights/best.pt')

## Celda 7 — Ver gráficas del entrenamiento

In [ ]:
from IPython.display import Image as IPImage
import glob

for img_path in glob.glob('/content/drive/MyDrive/EPP_Dataset/modelo_epp_v3/*.png'):
    print(img_path)
    display(IPImage(img_path, width=800))

## Celda 8 — Validar el modelo con imágenes de prueba

In [ ]:
from ultralytics import YOLO
from IPython.display import Image as IPImage
import glob, random, os

model = YOLO('/content/drive/MyDrive/EPP_Dataset/modelo_epp_v3/weights/best.pt')

test_imgs = glob.glob(f'{DATASET_DIR}/test/images/*.jpg')
muestra   = random.sample(test_imgs, min(5, len(test_imgs)))

os.makedirs('/content/predicciones', exist_ok=True)
for img_path in muestra:
    r = model(img_path, conf=0.4)
    r[0].save(filename=f'/content/predicciones/{os.path.basename(img_path)}')

for pred_img in glob.glob('/content/predicciones/*.jpg'):
    display(IPImage(pred_img, width=640))

## Celda 9 — Ver métricas por clase (mAP50)
Verificar especialmente `no-helmet` — debería superar 60% con el dataset fusionado.

In [ ]:
from ultralytics import YOLO

model   = YOLO('/content/drive/MyDrive/EPP_Dataset/modelo_epp_v3/weights/best.pt')
metrics = model.val(data=YAML_PATH, device=0)

names = ['helmet', 'no-helmet', 'vest', 'no-vest']
print(f'\n{"Clase":<15} {"mAP50":>8}')
print('-' * 25)
for i, name in enumerate(names):
    try:
        print(f'{name:<15} {metrics.box.maps[i]:>8.1%}')
    except Exception:
        print(f'{name:<15} {"N/A":>8}')
print(f'{"PROMEDIO":<15} {metrics.box.map50:>8.1%}')

## Celda 10 — Descargar el modelo entrenado
El modelo se guardó en Google Drive en:
`EPP_Dataset/modelo_epp_v3/weights/best.pt`

Descargarlo y reemplazar `best.pt` en la carpeta del proyecto.

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/EPP_Dataset/modelo_epp_v3/weights/best.pt')